# recs_013 — Heuristic ranker candidates on `two_tower_v1` pools

**Official D1 exploration notebook (A–G).**

**Train/tune on train pools · evaluate on val (no leakage).**

**Decision note:** Candidate **C** (`two_tower_v1_heuristic_logpop_blend`, `alpha=0.2`) is the promoted D1 winner from this notebook. This notebook is the single source of truth for D1.

**Candidate guide:** [`recs_013_ranker_d1_heuristic_candidates_learn.ipynb`](recs_013_ranker_d1_heuristic_candidates_learn.ipynb) — formulas, flavors, further reading.

## Notebook structure

1. **Imports + paths**
2. **Load data** — frozen train/val pools + catalog (`pop_row`)
3. **Scoring functions** — Candidates A–G + shared rerank helper
4. **Train grid-search** — best hyperparams per candidate (Slice A NDCG@10 primary)
5. **Val face-off** — each candidate @ train-tuned params + retrieval baseline + oracle

## Pipeline (run once)

```bash
python scripts/recs_job_build_example_cohort.py configs/recs_job_build_example_cohort_train_ranker.json
python scripts/recs_job_export_retrieval_pools.py configs/recs_job_export_retrieval_pools_train_ranker.json
# val pools: eval_offline_examples.jsonl from recs_job_eval_retrieval.py
```

## Candidates in this notebook

| ID | Method suffix | Params tuned on train |
|----|---------------|------------------------|
| A | `heuristic_pop_blend` | `alpha` |
| B | `heuristic_rrf` | `k`, `w_r`, `w_p` |
| C | `heuristic_logpop_blend` | `alpha` |
| D | `heuristic_geom_blend` | `alpha` |
| E | `heuristic_pop_topm` | `alpha`, `M` |
| F | `heuristic_pop_gated` | `beta`, `tau` |
| G | `heuristic_pop_only` | none (sanity baseline) |

**Val discipline:** hyperparameters are chosen on **train only**. Val is scored once per method with those fixed values — we never pick the best `alpha` (or other knob) by looking at val metrics. That is what "fixed params, no tuning on val" means.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import (
    load_retrieval_pool_rows,
    load_retrieval_pools_jsonl,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    average_precision_at_k,
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "two_tower_v1"
K_FINAL = 10
MIN_REVIEW_CHARS = 30
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"

ALPHAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

for p in (TRAIN_POOLS_PARQUET, VAL_JSONL):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

print(f"TRAIN_POOLS={TRAIN_POOLS_PARQUET}")
print(f"VAL_JSONL={VAL_JSONL}")

TRAIN_POOLS=/home/ryanr/workspace/steam_recommendations/artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet
VAL_JSONL=/home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [2]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

print(f"train pools: {len(train_pools):,}  val pools: {len(val_pools):,}  catalog apps: {len(app_ids):,}")

train pools: 51,691  val pools: 12,500  catalog apps: 315


## Scoring helpers (Candidates A–G)

In [3]:
def minmax_norm(x: np.ndarray) -> np.ndarray:
    """Per-pool min–max scale to [0, 1].

    For vector x over items in one retrieval pool:

        norm(x)_i = (x_i - min(x)) / (max(x) - min(x))

    If all values are equal, returns zeros (that signal cannot change rank order).
    Used so retrieval scores and popularity are on comparable scale before blending.
    """
    x = np.asarray(x, dtype=np.float64)
    lo, hi = float(x.min()), float(x.max())
    if hi - lo <= 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def pool_pop_values(pool_app_ids: list[int], *, pop_row: np.ndarray, app_to_row: dict[int, int]) -> np.ndarray:
    """Global train popularity for each app in the pool.

    pop_i = count of positive train-split reviews for game i (from ``pop_row``).
    Not normalized here — each candidate applies its own transform (raw, log, min-max).
    """
    return np.asarray([float(pop_row[app_to_row[int(a)]]) for a in pool_app_ids], dtype=np.float64)


def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    app_ids: np.ndarray,
    app_to_row: dict[int, int],
    k_final: int,
) -> np.ndarray:
    """Map pool scores → top-k catalog row indices (ranking @ k_final).

    Build a full-catalog score vector (non-pool apps = -inf), argsort descending,
    take first ``k_final``. Matches offline eval metric contract.
    """
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def score_a_blend(
    pool_app_ids: list[int], retrieval_scores: list[float], *, alpha: float, pop_row, app_to_row
) -> np.ndarray:
    """Candidate A — linear blend of normalized retrieval score and popularity.

    Let r = norm(retrieval_score), p = norm(pop) within this pool. Then:

        score = alpha * r + (1 - alpha) * p

    alpha=1 → retrieval order; alpha=0 → popularity order within pool.
    """
    pops = pool_pop_values(pool_app_ids, pop_row=pop_row, app_to_row=app_to_row)
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    pop = minmax_norm(pops)
    return alpha * retr + (1.0 - alpha) * pop


def score_b_rrf(
    pool_app_ids: list[int], retrieval_scores: list[float], *, k: float, w_r: float, w_p: float, pop_row, app_to_row
) -> np.ndarray:
    """Candidate B — reciprocal rank fusion (RRF).

    Rank each item by retrieval (rank_r, 1=best) and by popularity (rank_p). Then:

        score = w_r / (k + rank_r) + w_p / (k + rank_p)

    Uses order only, not score magnitudes. ``k`` dampens how much top ranks dominate (often 60).
    """
    retr = np.asarray(retrieval_scores, dtype=np.float64)
    pops = pool_pop_values(pool_app_ids, pop_row=pop_row, app_to_row=app_to_row)
    rank_retr = np.argsort(np.argsort(-retr)) + 1
    rank_pop = np.argsort(np.argsort(-pops)) + 1
    return w_r / (k + rank_retr) + w_p / (k + rank_pop)


def score_c_logpop_blend(
    pool_app_ids: list[int], retrieval_scores: list[float], *, alpha: float, pop_row, app_to_row
) -> np.ndarray:
    """Candidate C — linear blend with log-compressed popularity.

    Same as A but p = norm(log(1 + pop)). Compresses head-game counts before min-max:

        score = alpha * norm(retr) + (1 - alpha) * norm(log(1 + pop))
    """
    pops = np.log1p(pool_pop_values(pool_app_ids, pop_row=pop_row, app_to_row=app_to_row))
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    pop = minmax_norm(pops)
    return alpha * retr + (1.0 - alpha) * pop


def score_d_geom_blend(
    pool_app_ids: list[int], retrieval_scores: list[float], *, alpha: float, pop_row, app_to_row
) -> np.ndarray:
    """Candidate D — geometric (multiplicative) blend.

    With r = norm(retr), p = norm(pop) and small eps for stability:

        score = (r + eps)^alpha * (p + eps)^(1 - alpha)

    Both signals must be non-trivial; one near zero pulls the product down (stricter than A).
    """
    pops = pool_pop_values(pool_app_ids, pop_row=pop_row, app_to_row=app_to_row)
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    pop = minmax_norm(pops)
    eps = 1e-9
    return np.power(retr + eps, alpha) * np.power(pop + eps, 1.0 - alpha)


def score_e_topm_blend(
    pool_app_ids: list[int],
    retrieval_scores: list[float],
    *,
    alpha: float,
    M: int,
    pop_row,
    app_to_row,
) -> np.ndarray:
    """Candidate E — blend only the top-M retrieval items.

    Start with raw retrieval scores. Let H = indices of top-M by retrieval.
    Apply Candidate A formula on H only; leave tail items at raw retrieval score:

        score_i = blend(i)  if i in H else retrieval_score_i

    Protects strong retrieval hits outside the reranked head.
    """
    retr = np.asarray(retrieval_scores, dtype=np.float64)
    scores = retr.copy()
    head = np.argsort(-retr)[: int(M)]
    head_apps = [pool_app_ids[i] for i in head]
    head_retr = retr[head].tolist()
    blend = score_a_blend(head_apps, head_retr, alpha=alpha, pop_row=pop_row, app_to_row=app_to_row)
    for idx, s in zip(head, blend):
        scores[int(idx)] = float(s)
    return scores


def score_f_gated(
    pool_app_ids: list[int], retrieval_scores: list[float], *, beta: float, tau: float, pop_row, app_to_row
) -> np.ndarray:
    """Candidate F — gated popularity boost.

    Let r = norm(retr), p = norm(pop). Start with score = r. If r >= tau, add pop boost:

        score = r + beta * p   if r >= tau
        score = r              otherwise

    Popularity only helps items retrieval already ranked mid-tier or better.
    """
    pops = pool_pop_values(pool_app_ids, pop_row=pop_row, app_to_row=app_to_row)
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    pop = minmax_norm(pops)
    scores = retr.copy()
    mask = retr >= float(tau)
    scores[mask] = retr[mask] + float(beta) * pop[mask]
    return scores


def score_g_pop_only(
    pool_app_ids: list[int], retrieval_scores: list[float], *, pop_row, app_to_row, **_ignored
) -> np.ndarray:
    """Candidate G — sort pool by raw global popularity only.

        score_i = pop_i

    Ignores retrieval score (pool membership still comes from retriever). Sanity baseline vs A–F.
    """
    return pool_pop_values(pool_app_ids, pop_row=pop_row, app_to_row=app_to_row)


def score_retrieval_only(_pool_app_ids: list[int], retrieval_scores: list[float], **_ignored) -> np.ndarray:
    """Retrieval baseline — no rerank; use frozen two-tower scores as-is."""
    return np.asarray(retrieval_scores, dtype=np.float64)


ScoreFn = Callable[..., np.ndarray]

CANDIDATES: list[dict[str, Any]] = [
    {
        "id": "A",
        "method": f"{POOL_METHOD}_heuristic_pop_blend",
        "score_fn": score_a_blend,
        "param_grid": [{"alpha": a} for a in ALPHAS],
    },
    {
        "id": "B",
        "method": f"{POOL_METHOD}_heuristic_rrf",
        "score_fn": score_b_rrf,
        "param_grid": [
            {"k": k, "w_r": wr, "w_p": wp}
            for k in (10.0, 60.0)
            for wr, wp in ((1.0, 1.0), (2.0, 1.0))
        ],
    },
    {
        "id": "C",
        "method": f"{POOL_METHOD}_heuristic_logpop_blend",
        "score_fn": score_c_logpop_blend,
        "param_grid": [{"alpha": a} for a in ALPHAS],
    },
    {
        "id": "D",
        "method": f"{POOL_METHOD}_heuristic_geom_blend",
        "score_fn": score_d_geom_blend,
        "param_grid": [{"alpha": a} for a in ALPHAS],
    },
    {
        "id": "E",
        "method": f"{POOL_METHOD}_heuristic_pop_topm",
        "score_fn": score_e_topm_blend,
        "param_grid": [{"alpha": a, "M": m} for a in ALPHAS for m in (20, 50, 100)],
    },
    {
        "id": "F",
        "method": f"{POOL_METHOD}_heuristic_pop_gated",
        "score_fn": score_f_gated,
        "param_grid": [
            {"beta": b, "tau": t}
            for b in (0.2, 0.5, 0.8)
            for t in (0.2, 0.4, 0.6)
        ],
    },
    {
        "id": "G",
        "method": f"{POOL_METHOD}_heuristic_pop_only",
        "score_fn": score_g_pop_only,
        "param_grid": [{}],
    },
]

print(f"Registered {len(CANDIDATES)} candidates")

Registered 7 candidates


## Train: grid-search each candidate (Slice A primary)

In [ ]:
def mean_ndcg_for_scorer(
    pools: list[dict],
    score_fn: ScoreFn,
    params: dict[str, Any],
    *,
    slice_a_only: bool,
) -> float:
    """Average NDCG@K over examples: score pool → top-K → compare to held-out positives.

    Used for hyperparameter search on train pools only.
    """
    vals: list[float] = []
    extra = {"pop_row": pop_row, "app_to_row": app_to_row}
    for row in pools:
        if slice_a_only and int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        blend = score_fn(pool_apps, ret_sc, **params, **extra)
        ranked = pool_scores_to_ranked_indices(
            pool_apps, blend, app_ids=app_ids, app_to_row=app_to_row, k_final=K_FINAL
        )
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


def tune_candidate(candidate: dict[str, Any], pools: list[dict]) -> tuple[dict[str, Any], pd.DataFrame]:
    """Grid-search ``candidate['param_grid']`` on train; return best params by Slice A NDCG."""
    rows: list[dict[str, Any]] = []
    for params in candidate["param_grid"]:
        rows.append(
            {
                "candidate": candidate["id"],
                **params,
                "train_NDCG_slice_a": mean_ndcg_for_scorer(
                    pools, candidate["score_fn"], params, slice_a_only=True
                ),
                "train_NDCG_all": mean_ndcg_for_scorer(
                    pools, candidate["score_fn"], params, slice_a_only=False
                ),
            }
        )
    grid = pd.DataFrame(rows).sort_values("train_NDCG_slice_a", ascending=False)
    best_row = grid.iloc[0]
    param_keys = [k for k in best_row.index if k not in ("candidate", "train_NDCG_slice_a", "train_NDCG_all")]
    best_params = {k: best_row[k] for k in param_keys}
    if "M" in best_params:
        best_params["M"] = int(best_params["M"])
    return best_params, grid


best_params_by_id: dict[str, dict[str, Any]] = {}
train_grids: dict[str, pd.DataFrame] = {}

for cand in CANDIDATES:
    best_params, grid = tune_candidate(cand, train_pools)
    best_params_by_id[cand["id"]] = best_params
    train_grids[cand["id"]] = grid
    print(f"Candidate {cand['id']} best train params: {best_params}")

train_summary = pd.DataFrame(
    [
        {
            "candidate": c["id"],
            "method": c["method"],
            "best_params": json.dumps(best_params_by_id[c["id"]]),
            "train_NDCG_slice_a": train_grids[c["id"]].iloc[0]["train_NDCG_slice_a"],
        }
        for c in CANDIDATES
    ]
).sort_values("train_NDCG_slice_a", ascending=False)

display(Markdown("### Train tuning summary (best params per candidate)"))
display(train_summary)

Candidate A best train params: {'alpha': np.float64(0.2)}
Candidate B best train params: {'k': np.float64(60.0), 'w_r': np.float64(1.0), 'w_p': np.float64(1.0)}
Candidate C best train params: {'alpha': np.float64(0.2)}
Candidate D best train params: {'alpha': np.float64(0.2)}


## Val: face-off with train-tuned params (no val tuning)

Each candidate uses the params locked in above. We do **not** re-grid-search on val.

In [ ]:
def per_example_metrics(
    row: dict,
    *,
    method: str,
    score_fn: ScoreFn | None = None,
    params: dict[str, Any] | None = None,
    oracle: bool = False,
) -> dict:
    """Hit/MAP/NDCG/MRR for one val example.

    oracle=True: perfect reorder within the frozen top-100 pool (upper bound).
    score_fn=None: retrieval baseline (raw two-tower scores).
    else: candidate rerank with train-tuned ``params`` (fixed on val — no tuning).
    """
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)

    if oracle:
        ranked = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL]
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(
            pool_apps,
            score_retrieval_only(pool_apps, ret_sc),
            app_ids=app_ids,
            app_to_row=app_to_row,
            k_final=K_FINAL,
        )
    else:
        blend = score_fn(pool_apps, ret_sc, **(params or {}), pop_row=pop_row, app_to_row=app_to_row)
        ranked = pool_scores_to_ranked_indices(
            pool_apps, blend, app_ids=app_ids, app_to_row=app_to_row, k_final=K_FINAL
        )

    return {
        "method": method,
        "slice_name": row["slice_name"],
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
    }


val_rows: list[dict] = []
for row in val_pools:
    if not json.loads(row["validation_positive_app_ids_json"]):
        continue
    val_rows.append(per_example_metrics(row, method=POOL_METHOD, score_fn=None))
    for cand in CANDIDATES:
        val_rows.append(
            per_example_metrics(
                row,
                method=cand["method"],
                score_fn=cand["score_fn"],
                params=best_params_by_id[cand["id"]],
            )
        )
    val_rows.append(
        per_example_metrics(row, method=f"{POOL_METHOD}_oracle", oracle=True)
    )

df_val = pd.DataFrame(val_rows)

display(Markdown("### Val ranking overall"))
display(
    df_val.groupby("method")[["Hit@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values("NDCG@K", ascending=False)
)
display(Markdown("### Val ranking by slice"))
display(
    df_val.groupby(["slice_name", "method"])[["Hit@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values(["slice_name", "NDCG@K"], ascending=[True, False])
)

### Val ranking overall

,Hit@K,MAP@K,NDCG@K,MRR
method,,,,
two_tower_v1_oracle,0.51224,0.494053,0.498310,0.512240
two_tower_v1_heuristic_logpop_blend,0.19328,0.064321,0.092892,0.067059
two_tower_v1_heuristic_geom_blend,0.18784,0.060805,0.089036,0.063431
two_tower_v1_heuristic_pop_topm,0.18704,0.059078,0.087335,0.061505
two_tower_v1_heuristic_pop_blend,0.18704,0.059078,0.087335,0.061505
two_tower_v1_heuristic_pop_only,0.18120,0.057426,0.084670,0.059746
two_tower_v1_heuristic_rrf,0.16952,0.059396,0.083929,0.062289
two_tower_v1_heuristic_pop_gated,0.10000,0.031363,0.046607,0.032664
two_tower_v1,0.04680,0.010325,0.018161,0.011008


### Val ranking by slice

Hit@K    NDCG@K  \
slice_name            method                                                    
slice_a_multi_target  two_tower_v1_oracle                  0.773793  0.533618   
                      two_tower_v1_heuristic_rrf           0.245517  0.070246   
                      two_tower_v1_heuristic_logpop_blend  0.271724  0.068322   
                      two_tower_v1_heuristic_geom_blend    0.263448  0.066283   
                      two_tower_v1_heuristic_pop_blend     0.263448  0.061970   
                      two_tower_v1_heuristic_pop_topm      0.263448  0.061970   
                      two_tower_v1_heuristic_pop_only      0.256552  0.058621   
                      two_tower_v1_heuristic_pop_gated     0.142069  0.036108   
                      two_tower_v1                         0.091034  0.020537   
slice_b_single_target two_tower_v1_oracle                  0.496136  0.496136   
                      two_tower_v1_heuristic_logpop_blend  0.188450  0.094404   
                      two_tower_v1_heuristic_geom_blend    0.183185  0.090437   
                      two_tower_v1_heuristic_pop_blend     0.182335  0.088897   
                      two_tower_v1_heuristic_pop_topm      0.182335  0.088897   
                      two_tower_v1_heuristic_pop_only      0.176561  0.086274   
                      two_tower_v1_heuristic_rrf           0.164841  0.084772   
                      two_tower_v1_heuristic_pop_gated     0.097410  0.047253   
                      two_tower_v1                         0.044076  0.018015   

                                                                MRR  
slice_name            method                                         
slice_a_multi_target  two_tower_v1_oracle                  0.773793  
                      two_tower_v1_heuristic_rrf           0.088219  
                      two_tower_v1_heuristic_logpop_blend  0.081396  
                      two_tower_v1_heuristic_geom_blend    0.077948  
                      two_tower_v1_heuristic_pop_blend     0.070866  
                      two_tower_v1_heuristic_pop_topm      0.070866  
                      two_tower_v1_heuristic_pop_only      0.066864  
                      two_tower_v1_heuristic_pop_gated     0.040583  
                      two_tower_v1                         0.021192  
slice_b_single_target two_tower_v1_oracle                  0.496136  
                      two_tower_v1_heuristic_logpop_blend  0.066176  
                      two_tower_v1_heuristic_geom_blend    0.062537  
                      two_tower_v1_heuristic_pop_blend     0.060928  
                      two_tower_v1_heuristic_pop_topm      0.060928  
                      two_tower_v1_heuristic_pop_only      0.059307  
                      two_tower_v1_heuristic_rrf           0.060692  
                      two_tower_v1_heuristic_pop_gated     0.032177  
                      two_tower_v1                         0.010381